# disclosure v1

> disclosure provides progressive discovery of code messages for AI consumption.

In [ ]:
#| export
#| hide
import copy, re, json, ast, textwrap
from pathlib import Path
from dialoghelper.core import find_msgs

In [ ]:
#| hide
from fastcore.test import *
from fastcore.py2pyi import _proc_patches, has_deco
from pote.dutil import link_msg

## progressive code disclosure exploration

Yes, all good points. There is another somewhat easy reduction to implement (using AST or inspect), strip function bodys and leave just the signature + docments + dostring.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great idea — a "skeleton" mode for code cells. AST is the right tool: `ast.parse` the source, walk `FunctionDef`/`AsyncFunctionDef` nodes, and replace the body with just the docstring (if present) + `...`.

One design question before coding: should it apply to all functions in a cell, or only top-level ones? Nested functions/methods inside classes are less useful as stubs but could still matter.

<details class='token-usage-details'><summary>$0.0288</summary>

`total=50,428 | in=50,271 | out=157 | cached=98.9% | cache_new=311 | searches=0 | $0.0288`

</details>


Top level function/methods, inline/nested will be stripped anyway. In fact, we could also remove "private" functions/methods enterely and leave only the "public" API signatures. If the model want to check the dunder functions can always do it in antoher turn/tool call.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good rules. One clarification before coding: **classes** — keep the class definition with public method skeletons (stripping private methods entirely), or strip the whole class body down to `...`?

<details class='token-usage-details'><summary>$0.0283</summary>

`total=50,649 | in=50,495 | out=154 | cached=99.1% | cache_new=197 | searches=0 | $0.0283`

</details>


Class skeleton with full "public" API.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

So the full skeleton rules are:

- **Module-level**: keep public functions (signature + docstring + `...`), drop private ones entirely
- **Classes**: keep class def + public methods (signature + docstring + `...`), drop private methods entirely
- **Nested functions**: dropped (they become `...` inside their parent's body)

One more: **class-level assignments** like `x: int = 5` or `_exports = {...}` — keep public ones, drop private, or drop all?

<details class='token-usage-details'><summary>$0.0272</summary>

`total=50,722 | in=50,604 | out=118 | cached=99.2% | cache_new=163 | searches=0 | $0.0272`

</details>


**class/module-level assignments**? don't know just yet. Let's leave them for now. Function/method body. Don't code just yet, please. We have to think how to modularize all this (and other message content operations) `filter_msgs_v1` is approaching rapidly spaguetti level quality. (the old version `filter_msgs` above had a good modularization with "scoped" vs "metada: filtering.)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good point. Let's think about the layers:

1. **Selection** — which messages to include (`filter_msgs_v1`: ids, section, msg_type, metadata, re_pattern)
2. **Transformation** — how to render each message's content (skeleton, clamp_note, strip outputs...)
3. **Serialization** — how to format the result (`exportmsgs`: xml, md, json + frontmatter)

Right now `filter_msgs_v1` is pure selection, which is clean. The skeleton/clamp logic lives in `exportmsgs`. The question is whether transformation stays in `exportmsgs` or becomes a separate pipeline step between selection and serialization.

A clean separation could be: `filter_msgs_v1` → `transform_msgs(msgs, mode='skeleton'|'clamp'|...)` → `exportmsgs`. Does that feel right?

<details class='token-usage-details'><summary>$0.0297</summary>

`total=51,080 | in=50,877 | out=203 | cached=99.0% | cache_new=182 | searches=0 | $0.0297`

</details>


Feels good. (Even `filter_msgs_v1` can also be modularized a bit more– extract "scope" toa helper. Not urgent as of now.)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Agreed — `_scoped_msgs` already exists from the old version and could slot right in when needed.

On the pipeline: `transform_msgs` would need to know *per message* what to apply. That suggests a small config object or a list of transformation specs, e.g.:

```python
transform_msgs(msgs, code='skeleton', notes='header++', outputs=False)
```

Does that signature feel right, or would you prefer something more composable — e.g. a list of per-type transform functions?

<details class='token-usage-details'><summary>$0.0286</summary>

`total=51,294 | in=51,174 | out=120 | cached=98.8% | cache_new=358 | searches=0 | $0.0286`

</details>


My intuition says `clamp_msg` with a single input with a semantic similar to `hdr_clamp` in `exportmsgs` header++ -> header+ -> header, that is, a progressively more aggresive "clamping" clamp level 1 -> clamp level 2 -> .... Maybe two, one for note, other for code, but I prefer a single axis. Remember this is for token budget control, not compaction or summarizing per se, so the focus should be in reducing token usage minimizing info loss. Proposals?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

A single axis with named levels, mapping to per-type rules:

| Level | Notes | Code source | Code output |
|---|---|---|---|
| `'full'` | full | full | included |
| `'trim'` | `header++` | full | stripped |
| `'skim'` | `header+` | skeleton (sig + docstring) | stripped |
| `'toc'` | `header` | sig only | stripped |

So `clamp_msg(m, 'skim')` applies the right rule depending on `m.msg_type`. One axis, progressive, no summarization involved — purely structural reduction.

Does that feel right, or would you tweak any of the level/rule mappings?

<details class='token-usage-details'><summary>$0.2987</summary>

`total=51,896 | in=51,474 | out=422 | cached=0.0% | cache_new=51,139 | searches=0 | $0.2987`

</details>


| Level | Notes | Code source | Code output |
|---|---|---|---|
| `'full'` | full | full | included |
| `'trim'` | `header++` | full | included |
| `'skim'` | `header+` | full | stripped |
| `'sig'` | `header+` | skeleton (sig + docstring) | stripped |
| `'toc'` | `header` | sig only | stripped |


Umm, perhaps with should have two knobs for note+code. I many cases is probably more important the notes+sig (the `what`) than the body of the functions (the `how`). Let's the caller decides which axis trim more.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good insight. So two independent axes:

```python
clamp_msg(m, note='header++', code='sig')
```

Each has its own progression:
- `note`: `'full'` → `'header++'` → `'header+'` → `'header'`
- `code`: `'full'` → `'trim'` (no output) → `'sig'` → `'toc'` (sig only, no docstring)

And output is implied by `code` level — `full` includes it, everything else strips it. Does that feel right?

<details class='token-usage-details'><summary>$0.2946</summary>

`total=52,046 | in=51,909 | out=137 | cached=0.0% | cache_new=51,490 | searches=0 | $0.2946`

</details>


- `code`: `'full'` → `'clean'` (no output) → `'trim'` (ipython ?) → `'sig'` (sig only, no docments/docstring)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Clean breakdown. What does `'trim'` strip beyond output — IPython magics (`!`, `%%`, `?`) and test cells (`test_eq`, `test_is`...)? Or something else?

<details class='token-usage-details'><summary>$0.0314</summary>

`total=52,361 | in=52,148 | out=213 | cached=98.7% | cache_new=401 | searches=0 | $0.0314`

</details>


In [ ]:
import inspect
del_msg??
del_msg?
inspect.signature(del_msg)


```python
@llmtool
async def del_msg(
    id:str=None, # id of message to delete
    dname:str='', # Dialog to get info for; defaults to current dialog
    log_changed:bool=False # Add a note showing the deleted content?
):
    "Delete a message from the dialog. DO NOT USE THIS unless you have been explicitly instructed to delete messages."
    if log_changed: msg = await read_msgid(id, dname=dname)
    res = await call_endpa('rm_msg_', dname, raiseex=True, msid=id, json=True)
    if log_changed: await add_msg(f"> Deleted {msg_ref(id, dname)}\n\n```\n{msg.content}\n```")
    return res
```

**File:** `/usr/local/lib/python3.12/site-packages/dialoghelper/core.py`

```python
async def del_msg(
    id:str=None, # id of message to delete
    dname:str='', # Dialog to get info for; defaults to current dialog
    log_changed:bool=False, # Add a note showing the deleted content?
):

```



```
Delete a message from the dialog. DO NOT USE THIS unless you have been explicitly instructed to delete messages.
```



**File:** `/usr/local/lib/python3.12/site-packages/dialoghelper/core.py`

**Type:** function

<Signature (id: str = None, dname: str = '', log_changed: bool = False)>

del_msg?? -> del_msg? + docsstrings > signature + docstring (no annotations or docments)

We can look at llms.txt facilities or even use inspect/IPython magics and don't need to parse ourselves, don't know.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good distinction. So `'trim'` = strip output only; `'sig'` = signature + docstring, no body, no annotations, no docments (the `# comment` style fastai uses for param docs).

On parsing: since we're working with *source text in cells* (not imported symbols), `inspect` won't help directly — it works on live objects. AST is the natural tool for source-level surgery. IPython `?` just calls `inspect` under the hood.

Unless you want to *execute* the cell first and then inspect the resulting live objects — but that's a different pipeline entirely. Stick with AST?

<details class='token-usage-details'><summary>$0.3056</summary>

`total=53,186 | in=52,755 | out=431 | cached=0.0% | cache_new=52,042 | searches=0 | $0.3056`

</details>


You're very right, AST it is. (But I think docments are very useful, we have to preverve them if possible). `llms.txt` CLI makes something similar to this, remember the proverbial wheel.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good point on docments — they're already compact tokens, worth keeping.

On `llms.txt`: I'm thinking of `nbdev`'s `nbdev_sidebar`/export tooling and Jeremy's llms.txt spec, but do you know if there's a specific CLI command in that ecosystem that does source → skeleton reduction? Worth checking before writing AST code.

<details class='token-usage-details'><summary>$0.0358</summary>

`total=53,328 | in=53,003 | out=325 | cached=98.2% | cache_new=702 | searches=0 | $0.0358`

</details>


Yes, all good points. There is another somewhat easy reduction to implement (using AST or inspect), strip function bodys and leave just the signature + docments + dostring.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great idea — a "skeleton" mode for code cells. AST is the right tool: `ast.parse` the source, walk `FunctionDef`/`AsyncFunctionDef` nodes, and replace the body with just the docstring (if present) + `...`.

One design question before coding: should it apply to all functions in a cell, or only top-level ones? Nested functions/methods inside classes are less useful as stubs but could still matter.

<details class='token-usage-details'><summary>$0.0288</summary>

`total=50,428 | in=50,271 | out=157 | cached=98.9% | cache_new=311 | searches=0 | $0.0288`

</details>


Top level function/methods, inline/nested will be stripped anyway. In fact, we could also remove "private" functions/methods enterely and leave only the "public" API signatures. If the model want to check the dunder functions can always do it in antoher turn/tool call.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good rules. One clarification before coding: **classes** — keep the class definition with public method skeletons (stripping private methods entirely), or strip the whole class body down to `...`?

<details class='token-usage-details'><summary>$0.0283</summary>

`total=50,649 | in=50,495 | out=154 | cached=99.1% | cache_new=197 | searches=0 | $0.0283`

</details>


Class skeleton with full "public" API.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

So the full skeleton rules are:

- **Module-level**: keep public functions (signature + docstring + `...`), drop private ones entirely
- **Classes**: keep class def + public methods (signature + docstring + `...`), drop private methods entirely
- **Nested functions**: dropped (they become `...` inside their parent's body)

One more: **class-level assignments** like `x: int = 5` or `_exports = {...}` — keep public ones, drop private, or drop all?

<details class='token-usage-details'><summary>$0.0272</summary>

`total=50,722 | in=50,604 | out=118 | cached=99.2% | cache_new=163 | searches=0 | $0.0272`

</details>


**class/module-level assignments**? don't know just yet. Let's leave them for now. Function/method body. Don't code just yet, please. We have to think how to modularize all this (and other message content operations) `filter_msgs_v1` is approaching rapidly spaguetti level quality. (the old version `filter_msgs` above had a good modularization with "scoped" vs "metada: filtering.)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good point. Let's think about the layers:

1. **Selection** — which messages to include (`filter_msgs_v1`: ids, section, msg_type, metadata, re_pattern)
2. **Transformation** — how to render each message's content (skeleton, clamp_note, strip outputs...)
3. **Serialization** — how to format the result (`exportmsgs`: xml, md, json + frontmatter)

Right now `filter_msgs_v1` is pure selection, which is clean. The skeleton/clamp logic lives in `exportmsgs`. The question is whether transformation stays in `exportmsgs` or becomes a separate pipeline step between selection and serialization.

A clean separation could be: `filter_msgs_v1` → `transform_msgs(msgs, mode='skeleton'|'clamp'|...)` → `exportmsgs`. Does that feel right?

<details class='token-usage-details'><summary>$0.0297</summary>

`total=51,080 | in=50,877 | out=203 | cached=99.0% | cache_new=182 | searches=0 | $0.0297`

</details>


Feels good. (Even `filter_msgs_v1` can also be modularized a bit more– extract "scope" toa helper. Not urgent as of now.)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Agreed — `_scoped_msgs` already exists from the old version and could slot right in when needed.

On the pipeline: `transform_msgs` would need to know *per message* what to apply. That suggests a small config object or a list of transformation specs, e.g.:

```python
transform_msgs(msgs, code='skeleton', notes='header++', outputs=False)
```

Does that signature feel right, or would you prefer something more composable — e.g. a list of per-type transform functions?

<details class='token-usage-details'><summary>$0.0286</summary>

`total=51,294 | in=51,174 | out=120 | cached=98.8% | cache_new=358 | searches=0 | $0.0286`

</details>


My intuition says `clamp_msg` with a single input with a semantic similar to `hdr_clamp` in `exportmsgs` header++ -> header+ -> header, that is, a progressively more aggresive "clamping" clamp level 1 -> clamp level 2 -> .... Maybe two, one for note, other for code, but I prefer a single axis. Remember this is for token budget control, not compaction or summarizing per se, so the focus should be in reducing token usage minimizing info loss. Proposals?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

A single axis with named levels, mapping to per-type rules:

| Level | Notes | Code source | Code output |
|---|---|---|---|
| `'full'` | full | full | included |
| `'trim'` | `header++` | full | stripped |
| `'skim'` | `header+` | skeleton (sig + docstring) | stripped |
| `'toc'` | `header` | sig only | stripped |

So `clamp_msg(m, 'skim')` applies the right rule depending on `m.msg_type`. One axis, progressive, no summarization involved — purely structural reduction.

Does that feel right, or would you tweak any of the level/rule mappings?

<details class='token-usage-details'><summary>$0.2987</summary>

`total=51,896 | in=51,474 | out=422 | cached=0.0% | cache_new=51,139 | searches=0 | $0.2987`

</details>


| Level | Notes | Code source | Code output |
|---|---|---|---|
| `'full'` | full | full | included |
| `'trim'` | `header++` | full | included |
| `'skim'` | `header+` | full | stripped |
| `'sig'` | `header+` | skeleton (sig + docstring) | stripped |
| `'toc'` | `header` | sig only | stripped |


Umm, perhaps with should have two knobs for note+code. I many cases is probably more important the notes+sig (the `what`) than the body of the functions (the `how`). Let's the caller decides which axis trim more.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good insight. So two independent axes:

```python
clamp_msg(m, note='header++', code='sig')
```

Each has its own progression:
- `note`: `'full'` → `'header++'` → `'header+'` → `'header'`
- `code`: `'full'` → `'trim'` (no output) → `'sig'` → `'toc'` (sig only, no docstring)

And output is implied by `code` level — `full` includes it, everything else strips it. Does that feel right?

<details class='token-usage-details'><summary>$0.2946</summary>

`total=52,046 | in=51,909 | out=137 | cached=0.0% | cache_new=51,490 | searches=0 | $0.2946`

</details>


- `code`: `'full'` → `'clean'` (no output) → `'trim'` (ipython ?) → `'sig'` (sig only, no docments/docstring)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Clean breakdown. What does `'trim'` strip beyond output — IPython magics (`!`, `%%`, `?`) and test cells (`test_eq`, `test_is`...)? Or something else?

<details class='token-usage-details'><summary>$0.0314</summary>

`total=52,361 | in=52,148 | out=213 | cached=98.7% | cache_new=401 | searches=0 | $0.0314`

</details>


In [ ]:
import inspect
del_msg??
del_msg?
inspect.signature(del_msg)


```python
@llmtool
async def del_msg(
    id:str=None, # id of message to delete
    dname:str='', # Dialog to get info for; defaults to current dialog
    log_changed:bool=False # Add a note showing the deleted content?
):
    "Delete a message from the dialog. DO NOT USE THIS unless you have been explicitly instructed to delete messages."
    if log_changed: msg = await read_msgid(id, dname=dname)
    res = await call_endpa('rm_msg_', dname, raiseex=True, msid=id, json=True)
    if log_changed: await add_msg(f"> Deleted {msg_ref(id, dname)}\n\n```\n{msg.content}\n```")
    return res
```

**File:** `/usr/local/lib/python3.12/site-packages/dialoghelper/core.py`

```python
async def del_msg(
    id:str=None, # id of message to delete
    dname:str='', # Dialog to get info for; defaults to current dialog
    log_changed:bool=False, # Add a note showing the deleted content?
):

```



```
Delete a message from the dialog. DO NOT USE THIS unless you have been explicitly instructed to delete messages.
```



**File:** `/usr/local/lib/python3.12/site-packages/dialoghelper/core.py`

**Type:** function

<Signature (id: str = None, dname: str = '', log_changed: bool = False)>

del_msg?? -> del_msg? + docsstrings > signature + docstring (no annotations or docments)

We can look at llms.txt facilities or even use inspect/IPython magics and don't need to parse ourselves, don't know.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good distinction. So `'trim'` = strip output only; `'sig'` = signature + docstring, no body, no annotations, no docments (the `# comment` style fastai uses for param docs).

On parsing: since we're working with *source text in cells* (not imported symbols), `inspect` won't help directly — it works on live objects. AST is the natural tool for source-level surgery. IPython `?` just calls `inspect` under the hood.

Unless you want to *execute* the cell first and then inspect the resulting live objects — but that's a different pipeline entirely. Stick with AST?

<details class='token-usage-details'><summary>$0.3056</summary>

`total=53,186 | in=52,755 | out=431 | cached=0.0% | cache_new=52,042 | searches=0 | $0.3056`

</details>


You're very right, AST it is. (But I think docments are very useful, we have to preverve them if possible). `llms.txt` CLI makes something similar to this, remember the proverbial wheel.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good point on docments — they're already compact tokens, worth keeping.

On `llms.txt`: I'm thinking of `nbdev`'s `nbdev_sidebar`/export tooling and Jeremy's llms.txt spec, but do you know if there's a specific CLI command in that ecosystem that does source → skeleton reduction? Worth checking before writing AST code.

<details class='token-usage-details'><summary>$0.0358</summary>

`total=53,328 | in=53,003 | out=325 | cached=98.2% | cache_new=702 | searches=0 | $0.0358`

</details>


## references for code disclosure

Let's do this. I'm going to generate with `fitler_msgs_v1` and `exportmsgs` a markdown that should serve us to jump start a new discussion easily with the adecuate context.

In [ ]:
# await add_msg(read_url('https://llmstxt.org/intro.html.md'), msg_type='raw', i_collapsed=1);

In [ ]:
# await add_msg(read_url('https://llmstxt.org/core.html.md'), msg_type='raw', i_collapsed=1);

In [ ]:
# await add_msg(read_url('https://raw.githubusercontent.com/AnswerDotAI/llms-txt/refs/heads/main/llms_txt/core.py'), msg_type='raw', i_collapsed=1);

In [ ]:
# await add_msg(read_url('https://answerdotai.github.io/pysymbol-llm/index.html.md'), msg_type='raw', i_collapsed=1);

In [ ]:
# await add_msg(read_url('https://answerdotai.github.io/pysymbol-llm/core.html.md'), msg_type='raw', i_collapsed=1);

In [ ]:
# await add_msg(read_url('https://github.com/AnswerDotAI/pysymbol-llm/blob/main/pysymbol_llm/core.py'), msg_type='raw', i_collapsed=1);

This is how I generate `apilist.txt`, `llms-ctx.txt` and `llms-ctx-full.txt` for this project `pote`

```bash
#!/bin/bash

echo "Refreshing LLM documentation files..."

cd nbs || exit 1

echo "Generating API list documentation..."
pysym2md pote --output_file apilist.txt

echo "Generating context files..."
llms_txt2ctx llms.txt > llms-ctx.txt
llms_txt2ctx llms.txt --optional True > llms-ctx-full.txt

cd ..

echo "✅ Documentation refresh complete!"
```

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`pysymbol-llm` uses `astroid` (not stdlib `ast`) to walk a *live imported module*, not source text — so it won't work directly on cell source strings. But the logic is instructive: `process_function` and `process_class` do exactly what we need.

The key difference: we'll parse cell source text with `ast`, not import a module. The skeleton rules map cleanly to AST node surgery we can implement ourselves.

Ready to start?

<details class='token-usage-details'><summary>$0.3339</summary>

`total=60,521 | in=60,410 | out=111 | cached=0.0% | cache_new=52,934 | searches=0 | $0.3339`

</details>


Ready.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let's start with the simplest piece: a function that takes a single `ast.FunctionDef` node and returns its skeleton as source text — signature + docments + docstring + `...`. 

What should we use to unparse the signature back to text? `ast.unparse` gives us the whole function, but we only want the `def` line. We could reconstruct it manually from the node, or use `ast.unparse` on a modified node with a stub body. Which feels cleaner to you?

<details class='token-usage-details'><summary>$0.3454</summary>

`total=60,793 | in=60,578 | out=215 | cached=0.0% | cache_new=60,366 | searches=0 | $0.3454`

</details>


In [ ]:
# import ipykernel_helper.core
# ipykernel_helper.core??

## Code message disclosure v1
Progressive disclosure of Python source code at three levels of detail: signatures only (level 1), signatures with docstrings (level 2), and full source (level 3). Works cell-by-cell, preserving message boundaries and dialog narrative structure. Built on direct AST source reconstruction.

In [ ]:
#| export
def _sig_src(node, src):
    "Get original source lines for just the signature (decorators thru colon)"
    ls = src.splitlines()
    start = node.decorator_list[0].lineno if node.decorator_list else node.lineno
    body_line = node.body[0].lineno - 1
    if start - 1 == body_line: return ls[start-1][:node.body[0].col_offset].rstrip()
    return '\n'.join(ls[start-1:body_line])

def _node_doc(node): return ast.get_docstring(node) or ''

def _param_comments(node, lines):
    "Get comments for each parameter from source lines"
    comments = {}
    for arg in node.args.args + node.args.posonlyargs + node.args.kwonlyargs:
        line = lines[arg.lineno - 1]
        if '#' in line: comments[arg.arg] = line.split('#', 1)[1].strip()
    return comments

In [ ]:
_src = """
def add(
    a:int, # first number
    b:int=0, # second number
) -> int:
    "Sum two numbers."
    return a+b
"""
_tree2 = ast.parse(_src)
_fn = next(n for n in ast.walk(_tree2) if isinstance(n, ast.FunctionDef))
_sig = _sig_src(_fn, _src)
test_is('return a+b' in _sig, False)
test_eq(_node_doc(_fn), "Sum two numbers.")
test_eq(_param_comments(_fn, _src.splitlines()), {'a': 'first number', 'b': 'second number'})
_sig

'def add(\n    a:int, # first number\n    b:int=0, # second number\n) -> int:'

In [ ]:
#| export
_compound = (ast.If, ast.Try, ast.TryStar, ast.With, ast.For, ast.While)

def _subbodies(node):
    "Collect all child statement lists from a compound AST node"
    subs = []
    for attr in ('body', 'orelse', 'finalbody'): subs += getattr(node, attr, [])
    for h in getattr(node, 'handlers', []): subs += h.body
    return subs

def _stub(node, keep_doc=False):
    "Replace function/method bodies with `...`, recursing into class bodies"
    node = copy.deepcopy(node)
    if isinstance(node, ast.ClassDef):
        is_doc = lambda n: isinstance(n, ast.Expr) and isinstance(n.value, ast.Constant) and isinstance(n.value.value, str)
        node.body = [_stub(n, keep_doc) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) else n
                     for n in node.body if keep_doc or not is_doc(n)]
    else:
        doc = [node.body[0]] if keep_doc and ast.get_docstring(node) else []
        node.body = doc + [ast.Expr(value=ast.Constant(value=...))]
    return node

def _compact(s): return re.sub(r'\n{2,}', '\n', s).strip()

In [ ]:
#| export
def _fmt_doc(doc, ind):
    "Format docstring with `ind` indentation"
    if '\n' in doc: return f'{ind}"""{doc.replace(chr(10), chr(10)+ind)}"""'
    return f'{ind}"""{doc}"""' if '"' in doc else f'{ind}"{doc}"'

def _fmt_node(node, src, lines, level):
    "Format a single AST node at disclosure `level`"
    if level == 3:
        start = node.decorator_list[0].lineno-1 if node.decorator_list else node.lineno-1
        return '\n'.join(lines[start:node.end_lineno])
    if level == 1: return re.sub(r':\n +\.\.\.', ':...', ast.unparse(_stub(node)))
    sig = _sig_src(node, src)
    doc = _node_doc(node)
    one_liner = node.body[0].lineno == node.lineno
    ind = ' ' * (node.col_offset + 4 if one_liner else node.body[0].col_offset)
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        return f'{sig}...' if not doc else f'{sig}\n{_fmt_doc(doc, ind)}\n{ind}...'
    hdr = f'{sig}\n{_fmt_doc(doc, ind)}' if doc else sig
    return '\n'.join(filter(None, [hdr] + _fmt_body(node.body, src, lines, level)))

def _fmt_compound(node, src, lines, level):
    "Format a compound AST node (if/try/with/for/while) at disclosure level"
    ind = '    ' * (node.col_offset // 4 + 1)
    def _indent(parts): return '\n'.join(ind + l for p in parts for l in textwrap.dedent(p).splitlines())
    if isinstance(node, ast.If):
        parts = [f'if {ast.unparse(node.test)}:', _indent(_fmt_body(node.body, src, lines, level))]
        if node.orelse: parts += ['else:', _indent(_fmt_body(node.orelse, src, lines, level))]
        return '\n'.join(filter(None, parts))
    return '\n'.join(_fmt_body(_subbodies(node), src, lines, level))

def _fmt_body(nodes, src, lines, level):
    "Format AST body nodes at disclosure level, returning list of source segments"
    parts = []
    for node in nodes:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            parts.append(_fmt_node(node, src, lines, level))
        elif isinstance(node, (ast.Assign, ast.AnnAssign, ast.TypeAlias)):
            seg = ast.get_source_segment(src, node)
            if seg: parts.append(' ' * node.col_offset + seg)
        elif isinstance(node, _compound):
            parts.append(_fmt_compound(node, src, lines, level))
    return parts

In [ ]:
#| export
def disclose(src:str, level=1):
    "Return disclosure of `src` at `level` (1=sigs, 2=+docs/params, 3=full source)"
    tree, lines = ast.parse(src), src.splitlines()
    parts = []
    if tree.body and isinstance(tree.body[0], ast.Expr) and isinstance(tree.body[0].value, ast.Constant) and level > 1:
        parts.append(f'"""{tree.body[0].value.value}"""')
    if level > 1:
        imps = [ast.get_source_segment(src, n) for n in tree.body if isinstance(n, (ast.Import, ast.ImportFrom))]
        if imps: parts.append('\n'.join(imps))
    parts += _fmt_body(tree.body, src, lines, level)
    return _compact('\n\n'.join(filter(None, parts)))

### tests

In [ ]:
test_eq(disclose('', 1), ''); test_eq(disclose('', 2), ''); test_eq(disclose('', 3), '')
test_eq(disclose('(1+2)', 1), ''); test_eq(disclose('(1+2)', 2), ''); test_eq(disclose('(1+2)', 3), '')

test_eq(disclose("def bar(): pass", 1), "def bar():...")
test_eq(disclose("def bar(): pass", 2), "def bar():...")
test_eq(disclose("def bar(): pass", 3), "def bar(): pass")

In [ ]:
_s_mods = "x = 5\ny:int = 10\ntype Alias = list[int]"

test_eq(disclose(_s_mods, 1), 'x = 5\ny:int = 10\ntype Alias = list[int]')
test_eq(disclose(_s_mods, 2), disclose(_s_mods, 1))
test_eq(disclose(_s_mods, 3), disclose(_s_mods, 1))

In [ ]:
_s_moddoc = '"""This module does X."""\n\ndef foo(): pass'

test_eq(disclose(_s_moddoc, 1), "def foo():...")
test_eq(disclose(_s_moddoc, 2), '"""This module does X."""\ndef foo():...')
test_eq(disclose(_s_moddoc, 3), '"""This module does X."""\ndef foo(): pass')

In [ ]:
_s_fn = "def foo(a:int, b:str='hi') -> bool: return True"

test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool:...")
test_eq(disclose(_s_fn, 3), "def foo(a:int, b:str='hi') -> bool: return True")

_s_async = "async def fetch(url:str) -> bytes:\n    'Fetch url.'\n    pass"

test_eq(disclose(_s_async, 1), "async def fetch(url: str) -> bytes:...")

_s_deco1 = "@patch\ndef foo(self:MyClass, x:int): pass"

test_eq(disclose(_s_deco1, 1), "@patch\ndef foo(self: MyClass, x: int):...")

_s_deco2 = "@log\n@patch\ndef foo(self:MyClass, x:int): pass"

test_eq(disclose(_s_deco2, 1), "@log\n@patch\ndef foo(self: MyClass, x: int):...")

In [ ]:
_s_doc = '''

def add(
    a:int, # first
    b:int=0, # second
) -> int:
    "Sum two numbers."
    return a+b
'''

test_eq(disclose(_s_doc, 1), "def add(a: int, b: int=0) -> int:...")
test_eq(disclose(_s_doc, 2), "def add(\n    a:int, # first\n    b:int=0, # second\n) -> int:\n    \"Sum two numbers.\"\n    ...")
test_eq(disclose(_s_doc, 3), _s_doc.strip())

test_is('return a+b' in disclose(_s_doc, 2), False)
test_is('return a+b' in disclose(_s_doc, 3), True)

In [ ]:
_s_cls = '''
class Foo(Base):
    "A foo."
    def __init__(self,
                 x:int, # x val
                 y:str='hi'): # y val
        "Init."
        self.x,self.y = x,y
    def bar(self) -> bool:
        "Check bar."
        return self.x > 0
    def _priv(self): pass
'''

_cls_l1 = 'class Foo(Base):\n    def __init__(self, x: int, y: str=\'hi\'):...\n    def bar(self) -> bool:...\n    def _priv(self):...'
_cls_l2 = 'class Foo(Base):\n    "A foo."\n    def __init__(self,\n                 x:int, # x val\n                 y:str=\'hi\'): # y val\n        "Init."\n        ...\n    def bar(self) -> bool:\n        "Check bar."\n        ...\n    def _priv(self):...'

test_eq(disclose(_s_cls, 1), _cls_l1)
test_eq(disclose(_s_cls, 2), _cls_l2)
test_eq(disclose(_s_cls, 3), _s_cls.strip())

In [ ]:
_s_tricky = '''
def foo():
    """
    Example::

        :\\n    ...

    ```python
    x = 1
    ```
    """
    pass

def bar(): pass

def baz():
    """Has "quotes" and \\'escaped\\' stuff."""
    pass
'''

_tricky_l2 = disclose(_s_tricky, 2)

test_eq('```python' in _tricky_l2, True)   # backticks in docstring survive
test_eq('pass' in disclose(_s_tricky, 3), True)  # level 3 preserves full bodies

_tricky_l2_bar = 'def bar():...'

test_eq(disclose('def bar(): pass', 2), _tricky_l2_bar)

In [ ]:
_s_dc = '''
from dataclasses import dataclass

@dataclass
class Point:
    "A 2D point."
    x: float
    y: float = 0.0
    def dist(self) -> float:
        "Distance from origin."
        return (self.x**2 + self.y**2)**0.5
'''

_dc_l1 = "@dataclass\nclass Point:\n    x: float\n    y: float = 0.0\n    def dist(self) -> float:..."
_dc_l2 = "from dataclasses import dataclass\n@dataclass\nclass Point:\n    \"A 2D point.\"\n    x: float\n    y: float = 0.0\n    def dist(self) -> float:\n        \"Distance from origin.\"\n        ..."

test_eq(disclose(_s_dc, 1), _dc_l1)
test_eq(disclose(_s_dc, 2), _dc_l2)
test_eq(disclose(_s_dc, 3), '\n'.join(filter(None, _s_dc.strip().splitlines())))

In [ ]:
_s_cls_assigns = '''
class Config:
    "App config."
    debug: bool = False
    MAX_RETRIES: int = 3
    name = "default"
    def reset(self): self.debug = False
'''

_cls_assigns_l1 = 'class Config:\n    debug: bool = False\n    MAX_RETRIES: int = 3\n    name = \'default\'\n    def reset(self):...'
_cls_assigns_l2 = 'class Config:\n    "App config."\n    debug: bool = False\n    MAX_RETRIES: int = 3\n    name = "default"\n    def reset(self):...'

test_eq(disclose(_s_cls_assigns, 1), disclose(_s_cls_assigns, 1))  # smoke test

test_eq(disclose(_s_cls_assigns, 1), _cls_assigns_l1)
test_eq(disclose(_s_cls_assigns, 2), _cls_assigns_l2)
test_eq(disclose(_s_cls_assigns, 3), _s_cls_assigns.strip())

In [ ]:
_s_cond = '''
import sys

if sys.version_info >= (3, 12):
    def new_feature(x:int) -> str: return str(x)
else:
    def new_feature(x:int) -> str: return repr(x)

def always_here(): pass
'''

_cond_l1 = "if sys.version_info >= (3, 12):\n    def new_feature(x: int) -> str:...\nelse:\n    def new_feature(x: int) -> str:...\ndef always_here():..."
_cond_l2 = "import sys\nif sys.version_info >= (3, 12):\n    def new_feature(x:int) -> str:...\nelse:\n    def new_feature(x:int) -> str:...\ndef always_here():..."

test_eq(disclose(_s_cond, 1), _cond_l1)
test_eq(disclose(_s_cond, 2), _cond_l2)
test_eq(disclose(_s_cond, 3), '\n'.join(filter(None, _s_cond.strip().splitlines())))

In [ ]:
_s_imp = '''
"""A module."""
import os, sys
from pathlib import Path

def foo(): pass
'''

_imp_l2 = '"""A module."""\nimport os, sys\nfrom pathlib import Path\ndef foo():...'
_imp_l3 = '"""A module."""\nimport os, sys\nfrom pathlib import Path\ndef foo(): pass'

test_eq(disclose(_s_imp, 1), "def foo():...")
test_eq(disclose(_s_imp, 2), _imp_l2)
test_eq(disclose(_s_imp, 3), _imp_l3)

### cell & notebook disclosure

In [ ]:
#| export
def disclose_cell(src:str, level=1):
    "Disclose a single cell's source at `level`; returns original if not parseable Python"
    try: return disclose(src, level)
    except SyntaxError: return src

In [ ]:
test_eq(disclose_cell("x = 1 +", 1), "x = 1 +")   # SyntaxError -> passthrough
test_eq(disclose_cell("# just a comment", 1), "")   # valid but empty -> ok
test_eq(disclose_cell(_s_fn, 1), disclose(_s_fn, 1))

In [ ]:
#| export
def _xml_cell(content, **attrs):
    attr_str = ''.join(f' {k}="{v}"' for k,v in attrs.items())
    return f'<cell{attr_str}>{content}</cell>'

async def disclose_dialog(dname:str='', level=1, only_exported=False, fmt='text'):
    "Return disclosed source of all code cells in dialog `dname` at `level`"
    msgs = await find_msgs(dname=dname, msg_type='code', include_output=False, only_exp=only_exported)
    parts = [(m['id'], disclose_cell(m['content'], level)) for m in msgs]
    parts = [(i,d) for i,d in parts if d]
    if fmt == 'xml':
        cells = '\n'.join(_xml_cell(d, id=i) for i,d in parts)
        return f'<dialog level="{level}" path="{dname or "."}">\n{cells}\n</dialog>'
    if fmt == 'json':
        return json.dumps(dict(level=level, path=dname or ".", cells=[dict(id=i, source=d) for i,d in parts]), indent=2)
    return _compact('\n\n'.join(d for _,d in parts))

def disclose_nb(path:str|Path, level=1, fmt='text'):
    "Return disclosed source of all code cells in notebook at `path` at `level`"
    nb = json.loads(Path(path).read_text())
    srcs = [c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            for c in nb['cells'] if c['cell_type'] == 'code']
    parts = [(i,d) for i,s in enumerate(srcs) if (d:=disclose_cell(s, level))]
    if fmt == 'xml':
        cells = '\n'.join(_xml_cell(d, id=i) for i,d in parts)
        return f'<nb level="{level}" path="{path}">\n{cells}\n</nb>'
    if fmt == 'json':
        return json.dumps(dict(level=level, path=str(path), cells=[dict(id=i, source=d) for i,d in parts]), indent=2)
    return _compact('\n\n'.join(d for _,d in parts))

Let's test the JSON format output to verify it works correctly:

In [ ]:
s = await disclose_dialog(only_exported=True, level=1, fmt='json')
print(s[:1000])

{
  "level": 1,
  "path": ".",
  "cells": [
    {
      "id": "_1fd56458",
      "source": "def _sig_src(node, src):...\ndef _node_doc(node):...\ndef _param_comments(node, lines):..."
    },
    {
      "id": "_804a5da7",
      "source": "_compound = (ast.If, ast.Try, ast.TryStar, ast.With, ast.For, ast.While)\ndef _subbodies(node):...\ndef _stub(node, keep_doc=False):...\ndef _compact(s):..."
    },
    {
      "id": "_2802fc51",
      "source": "def _fmt_doc(doc, ind):...\ndef _fmt_node(node, src, lines, level):...\ndef _fmt_compound(node, src, lines, level):...\ndef _fmt_body(nodes, src, lines, level):..."
    },
    {
      "id": "_a86e6423",
      "source": "def disclose(src: str, level=1):..."
    },
    {
      "id": "_8f48b1a5",
      "source": "def disclose_cell(src: str, level=1):..."
    },
    {
      "id": "_6e029abf",
      "source": "def _xml_cell(content, **attrs):...\nasync def disclose_dialog(dname: str='', level=1, only_exported=False, fmt='text'):...\ndef disclose_n

Let me test the XML format as well to verify all three formats work:

In [ ]:
s = await disclose_dialog(only_exported=True, level=2, fmt='xml')
print(s[:1500])

<dialog level="2" path=".">
<cell id="_1e28a183">import copy, re, json, ast, textwrap
from html import escape
from pathlib import Path
from typing import TypeAlias
from IPython import get_ipython
from mistletoe import Document
from mistletoe.block_token import CodeFence
from mistletoe.markdown_renderer import MarkdownRenderer
from nbdev.process import extract_directives
from fastcore.foundation import patch, adict, AttrDict as AD
from fastcore.meta import delegates
from fastcore.py2pyi import _body_ellip, functypes
from dialoghelper.core import add_msg, read_msg, find_msgs, msg_idx, run_msg, toggle_header, ast_py, find_dname
from pote.dutil import in_dialog</cell>
<cell id="_1fd56458">def _sig_src(node, src):
    "Get original source lines for just the signature (decorators thru colon)"
    ...
def _node_doc(node):...
def _param_comments(node, lines):
    "Get comments for each parameter from source lines"
    ...</cell>
<cell id="_804a5da7">_compound = (ast.If, ast.Try, ast.TryStar, a

This section implements a **progressive disclosure** system for Python source code — it lets you reveal code at varying levels of detail rather than always showing everything.

The core is `disclose(src, level)` which takes Python source and returns a summarised version at one of three levels:

- **Level 1** — signatures only (e.g. `def foo(x: int) -> bool:...`)
- **Level 2** — signatures + docstrings + parameter inline comments, but still no bodies
- **Level 3** — full original source, unchanged

The helper functions build up to this:
- `_sig_src` extracts just the signature lines (decorators through the colon)
- `_node_doc` / `_param_comments` pull out docstrings and `# inline` param comments
- `_stub` replaces function bodies with `...`, recursing into classes
- `_fmt_*` functions handle formatting at each level for functions, classes, and compound statements (`if`/`try`/`for` etc.)

Then `disclose_cell`, `disclose_dialog`, and `disclose_nb` wrap it to operate on a single cell, a whole solveit dialog, or a notebook file respectively — with output in plain text, XML, or JSON.

The practical use case is giving an AI a compact, token-efficient view of a codebase's API — enough to understand the structure and interfaces without drowning in implementation details.

List of the implicit assumptions that govern the implementation, the implicit rules hardcoded into the Python functions.

---

**Disclosure levels are strictly 1, 2, 3**
No interpolation, no partial levels. Passing anything else (e.g. `0` or `4`) falls through to level-1 or level-3 behaviour silently without error.

---

**Level 1 normalises, level 2 preserves**
Level 1 routes through `ast.unparse`, which reformats signatures (e.g. `a:int` → `a: int`, spaces added around operators). Level 2 routes through `_sig_src` which lifts the original source lines verbatim, so formatting is preserved exactly.

---

**Blank lines are always collapsed**
`_compact` runs on the final output at every level including 3. Blank lines between top-level definitions are always removed, even when returning "full source".

---

**Module docstring only shown at level ≥ 2**
And only if the very first node in the file is a string-constant expression. If the file starts with an import, no module docstring is detected even if one exists lower down.

---

**Imports only shown at level ≥ 2, and only top-level ones**
Only `ast.Import` / `ast.ImportFrom` nodes directly in `tree.body` are collected. Imports inside functions or classes are ignored.

---

**Bare expressions are silently dropped at all levels**
`(1+2)`, a lone variable reference, a `print(...)` call — all invisible. Only functions, classes, assignments, type aliases, and recognised compound statements survive.

---

**Only three assignment forms are preserved**
`ast.Assign`, `ast.AnnAssign`, `ast.TypeAlias`. Other statement types at module or class level (e.g. `raise`, `assert`, `del`) are silently dropped.

---

**Compound statement interiors are partially flattened**
`ast.If` is handled specially: it reconstructs `if <condition>:` / `else:` using `ast.unparse` on the test, then recurses into each branch. All other compound forms (`for`, `while`, `with`, `try`) are silently collapsed — their headers disappear entirely, and only the contents of their sub-bodies are disclosed.

---

**`elif` is treated as `else`**
There's no special handling for `elif` chains. An `elif` appears in `node.orelse` and will be formatted as `else:`.

---

**4-space indentation is assumed throughout**
`_fmt_compound` computes indentation as `'    ' * (node.col_offset // 4 + 1)`. Code using 2-space or tab indentation would produce wrong output.

---

**Parameter comments must be on the same line as the parameter**
`_param_comments` reads the line at `arg.lineno` and splits on the first `#`. A `#` inside a default-value string on a parameter line would be misread as a comment.

---

**`*args` and `**kwargs` param comments are never captured**
`_param_comments` only iterates `args.args`, `args.posonlyargs`, and `args.kwonlyargs`. Variadic and keyword-variadic arguments are excluded.

---

**Docstring quoting: `"` unless the doc contains `"`**
`_fmt_doc` uses a single `"` when safe, else falls back to `"""`. A docstring containing `"""` itself would produce invalid Python output — not handled.

---

**One-liner detection is line-number based**
A function is considered a one-liner if `node.body[0].lineno == node.lineno`. A function whose only statement is on the same line as `def` triggers the one-liner indentation path (`col_offset + 4`), otherwise `body[0].col_offset` is used.

---

**`_stub` does not recurse into compound statements inside class bodies**
Only `FunctionDef`, `AsyncFunctionDef`, and `ClassDef` children of a class body are recursed into. A method whose body is wrapped in an `if` would not be further examined.

---

**`disclose_cell` catches only `SyntaxError`**
Any other exception (e.g. a `MemoryError` or an AST library `ValueError`) propagates uncaught. On `SyntaxError` the original source is returned verbatim, not an empty string.

---

**XML output is unescaped**
`_xml_cell` does no escaping of cell content. Source containing `</cell>` would corrupt the XML structure.

---

**`disclose_nb` assumes nbformat source is `str` or `list[str]`**
It handles both by joining with `''`, covering old and new nbformat conventions, but makes no other format assumptions.

---

**`disclose_dialog` always excludes outputs**
`include_output=False` is hardcoded. Cell outputs are never part of the disclosure regardless of level.

---

**`fmt` parameter silently falls through to `'text'`**
In `disclose_dialog` / `disclose_nb`, an unrecognised `fmt` value skips the `xml` and `json` branches and produces text output without any warning.

## export -

In [ ]:
from pote.dutil import ctxusage
await ctxusage()

10109

In [ ]:
from pote.flakes import show_flakes
await show_flakes()

<div class="prose">

No warnings to report

</div>

In [ ]:
# #|hide
# #|eval: false
# from pote.dialog import dlg_export
# await dlg_export(solo_nb=True)